In [1]:
!uv pip install textstat

Resolved 10 packages in 950ms                                        
Prepared 6 packages in 853ms                                                 pyphen               ------------------------------ 2.02 MiB/2.02 MiB           pyphen               ------------------------------ 1.44 MiB/2.02 MiB           pyphen               ------------------------------ 703.89 KiB/2.02 MiB         pyphen               ------------------------------ 144.00 KiB/2.02 MiB         pyphen               ------------------------------ 95.70 KiB/2.02 MiB          
Installed 10 packages in 14ms                               
 + click==8.5.0
 + cloudpickle==3.1.2
 + defusedxml==0.7.1
 + joblib==1.6.0
 + nltk==3.10.3
 + pyphen==0.18.1
 + regex==2026.9.29
 + setuptools==84.0.0
 + textstat==0.7.13
 + tqdm==4.70.1


In [1]:
import textstat

# Define o idioma para Português
# textstat.set_lang('pt')

texto = "I'm in the car doing my homework above the dog but in amazing vibe"

# Calcula o score de Flesch
score = textstat.flesch_reading_ease(texto)

print(f"Score Flesch: {score:.2f}")

Score Flesch: 77.81


In [12]:
!uv pip install pyphen nltk

Checked 2 packages in 18ms


In [21]:
import re
import pyphen
import nltk

nltk.download('punkt', quiet=True)

# Dicionário de separação silábica oficial em PT-BR
dic_pt = pyphen.Pyphen(lang='pt_BR')

def calcular_flesch_portugues(texto: str) -> float:
    # 1. Limpeza simples do texto
    texto_limpo = re.sub(r'[^\w\s\.\!\?]', '', texto)
    
    # 2. Contagem de Frases
    frases = [f for f in re.split(r'[\.\!\?]+', texto_limpo) if f.strip()]
    total_frases = max(len(frases), 1)
    
    # 3. Contagem de Palavras
    palavras = [p for p in texto_limpo.split() if p.isalpha()]
    total_palavras = max(len(palavras), 1)
    
    # 4. Contagem Real de Sílabas em Português
    total_silabas = 0
    for palavra in palavras:
        # pyphen insere hífens nas sílabas (ex: "au-tis-mo" -> 3 partes)
        silabas = dic_pt.inserted(palavra).split('-')
        total_silabas += len(silabas)
        
    # 5. Variáveis da Fórmula PT
    cpb = total_palavras / total_frases   # Comprimento Médio da Frase
    pds = total_silabas / total_palavras   # Média de Sílabas por Palavra
    
    # 6. Aplicação da Fórmula Adaptada para Português (Martins et al.)
    flesch_score = 248.835 - (1.015 * cpb) - (84.6 * pds)
    
    return round(flesch_score, 2)

# --- Exemplo de Teste ---
texto_original = "Art. 1º Esta Lei institui a Política Nacional de Proteção dos Direitos da Pessoa com Transtorno do Espectro Autista e define diretrizes para sua consecução."
texto_simplificado = "Art. 1º Esta Lei cria a Política Nacional de Proteção dos Direitos da Pessoa com Autismo e define regras para colocar essa lei em prática."

print(f"Flesch Original (PT): {calcular_flesch_portugues(texto_original)}")
print(f"Flesch Simplificado (PT): {calcular_flesch_portugues(texto_simplificado)}")

Flesch Original (PT): 53.09
Flesch Simplificado (PT): 72.32


In [15]:
# --- Exemplo de Teste ---
texto_original = "Raio o dia com a contundência do explendor da girabrancia dos cometas intergalacticos"
texto_simplificado = "O dia amanheceu e o sol nasceu"

print(f"Flesch Original (PT): {calcular_flesch_portugues(texto_original)}")
print(f"Flesch Simplificado (PT): {calcular_flesch_portugues(texto_simplificado)}")

Flesch Original (PT): 53.42
Flesch Simplificado (PT): 120.87


In [23]:
# 1. Instale o spaCy e pyphen atualizados no seu ambiente uv/venv
!uv pip install -U spacy pyphen

# 2. Baixe o modelo oficial de Português atualizado (v3.x)
!uv pip install https://github.com/explosion/spacy-models/releases/download/pt_core_news_lg-3.7.0/pt_core_news_lg-3.7.0-py3-none-any.whl

Resolved 45 packages in 614ms                                        
Prepared 38 packages in 5.84s                                                spacy                ------------------------------ 33.22 MiB/33.85 MiB         spacy                ------------------------------ 10.65 MiB/33.85 MiB         spacy                ------------------------------ 3.72 MiB/33.85 MiB          spacy                ------------------------------ 1.16 MiB/33.85 MiB          spacy                ------------------------------ 142.83 KiB/33.85 MiB        spacy                ------------------------------ 78.83 KiB/33.85 MiB         spacy                ------------------------------ 62.83 KiB/33.85 MiB         spacy                ------------------------------ 30.83 KiB/33.85 MiB         mspacy                ------------------------------ 30.83 KiB/33.85 MiB         spacy                ------------------------------ 14.83 KiB/33.85 MiB         spacy                ------------------------------ 

In [ ]:
import spacy
import pyphen

# Carrega o modelo de Português
nlp = spacy.load("pt_core_news_lg")
dic_pt = pyphen.Pyphen(lang='pt_BR')

def calcular_flesch_spacy_v3(texto: str) -> dict:
    doc = nlp(texto)
    
    # Conta apenas frases reais
    frases = list(doc.sents)
    total_frases = max(len(frases), 1)
    
    # Filtra e conta apenas palavras reais (descarta pontuações e símbolos)
    palavras = [token.text for token in doc if token.is_alpha]
    total_palavras = max(len(palavras), 1)
    
    # Contagem exata de sílabas em PT-BR
    total_silabas = sum(len(dic_pt.inserted(p).split('-')) for p in palavras)
    
    cpb = total_palavras / total_frases    # Comprimento Médio da Frase
    pds = total_silabas / total_palavras    # Média de Sílabas por Palavra
    
    # Fórmula Adaptada para Português (Martins et al. / NILC)
    flesch_score = 248.835 - (1.015 * cpb) - (84.6 * pds)
    
    return {
        "flesch_score": round(flesch_score, 2),
        "total_palavras": total_palavras,
        "total_frases": total_frases
    }

# --- Teste de Execução ---
texto = "Art. 1º Esta Lei institui a Política Nacional de Proteção dos Direitos da Pessoa com Transtorno do Espectro Autista."
print(calcular_flesch_spacy_v3(texto))

In [24]:
!uv pip install git+https://github.com/NILC-NLP/nilc-metrix.git

   Updating https://github.com/NILC-NLP/nilc-metrix.git (HEAD)                  error: Git operation failed                                        
  Caused by: failed to clone into: /home/marco.santos/.cache/uv/git-v0/db/06bab046926be3ef
  Caused by: process didn't exit successfully: `/usr/bin/git fetch --force
             --update-head-ok 'https://github.com/NILC-NLP/nilc-metrix.git'
             '+HEAD:refs/remotes/origin/HEAD'` (exit status: 128)
    --- stderr
    fatal: could not read Username for 'https://github.com': terminal prompts
             disabled
